In [1]:
!pip install optuna mlflow dagshub seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.4/263.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import dagshub
import mlflow
import mlflow.sklearn

In [3]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6044376a-3ec3-4d26-963d-d27251bcaed7&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4fff1e978d7bd8d9a11170aaff3a18613ef4d4337ba6df3bd23ebb268384d2d5




Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [10]:
df = pd.read_csv("twitter_cleaned.csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [11]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [12]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (41292, 8000)
Test shape: (10323, 8000)


In [13]:
def build_logreg(trial):

    C = trial.suggest_float("C", 1e-3, 10.0, log=True)

    return LogisticRegression(
        C=C,
        solver="lbfgs",
        penalty="l2",
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

In [14]:
def objective(trial):

    model = build_logreg(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params({f"model_{k}": v for k, v in trial.params.items()})
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [15]:
study = optuna.create_study(
    direction="maximize",
    study_name="tfidf_logreg_study",
    storage="sqlite:///tfidf_logreg_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 09:58:55,461] Using an existing study with name 'tfidf_logreg_study' instead of creating a new one.


In [16]:
with mlflow.start_run(run_name="TFIDF_LogisticRegression"):

    # Log vectorizer parameters safely
    mlflow.log_param("model_name", "LogisticRegression")
    mlflow.log_param("vectorizer_type", "TfidfVectorizer")
    mlflow.log_param("vectorizer_ngram_range", "(1,2)")
    mlflow.log_param("vectorizer_max_features", 8000)
    mlflow.log_param("vectorizer_min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params({f"model_{k}": v for k, v in best_params.items()})
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = LogisticRegression(
        **best_params,
        solver="lbfgs",
        penalty="l2",
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X_train, y_train)

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = LogisticRegression(
            **best_params,
            solver="lbfgs",
            penalty="l2",
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_tfidf_logreg.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_tfidf_logreg.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - TFIDF LogisticRegression")
    plt.savefig("confusion_matrix_tfidf_logreg.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_tfidf_logreg.png")

    study.trials_dataframe().to_csv("optuna_trials_tfidf_logreg.csv", index=False)
    mlflow.log_artifact("optuna_trials_tfidf_logreg.csv")

    mlflow.sklearn.log_model(final_model, artifact_path="model")

print("TF-IDF + Logistic Regression experiment completed successfully.")

[I 2026-02-23 09:59:00,607] Trial 0 finished with value: 0.734679553849576 and parameters: {'C': 0.2744531725412781}. Best is trial 0 with value: 0.734679553849576.


🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/610fbbe5bc8541ce9b6c8a7843df2c03
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:59:06,682] Trial 1 finished with value: 0.8069815144618927 and parameters: {'C': 9.828051063481475}. Best is trial 1 with value: 0.8069815144618927.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/1c55a4d0dddc490b967d1a3e00b7b166
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
🏃 View run trial_2 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/cab7df60f72a4ac4b3ec291d605263b7
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:59:13,679] Trial 2 finished with value: 0.798538284596685 and parameters: {'C': 3.7237915278257647}. Best is trial 1 with value: 0.8069815144618927.


...

🏃 View run trial_49 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/8f3b74258d264350a0dd2f528a085c8c
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 10:04:42,667] Trial 49 finished with value: 0.7618750634745677 and parameters: {'C': 0.7556206648422553}. Best is trial 1 with value: 0.8069815144618927.


Best Macro F1: 0.8069815144618927
Best Params: {'C': 9.828051063481475}
Fold 1 - Macro F1: 0.8004
Fold 2 - Macro F1: 0.7981
Fold 3 - Macro F1: 0.7977
Fold 4 - Macro F1: 0.8046
Fold 5 - Macro F1: 0.7970
Final Test Macro F1: 0.8069815144618927


2026/02/23 10:05:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 10:05:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_LogisticRegression at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/1b36667889d541f6ad5704accc806ae8
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
TF-IDF + Logistic Regression experiment completed successfully.
